# Predictive Representativity analysis

This notebook translates the completed external validation results into an operational Predictive Representativity analysis.

It does not retrain or reevaluate models. It uses saved result tables from:

- `outputs/tables/all_models_all_seed_metrics.csv`
- `outputs/tables/all_models_all_seed_subgroup_metrics.csv`
- `outputs/tables/bosque_light_dark_bootstrap_comparison_readable.csv`

The goal is to quantify how far internal HAM10000 validation evidence is from external BOSQUE performance evidence, overall and by skin subgroup.

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

TABLES = ROOT / "outputs" / "tables"
FIGURES = ROOT / "outputs" / "figures"
PR_DIR = ROOT / "outputs" / "predictive_representativity"
PR_DIR.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT, "| exists:", ROOT.exists())
print("TABLES:", TABLES, "| exists:", TABLES.exists())
print("PR_DIR:", PR_DIR, "| exists:", PR_DIR.exists())

## 2. Load completed results

In [ ]:
overall = pd.read_csv(TABLES / "all_models_all_seed_metrics.csv")
subgroups = pd.read_csv(TABLES / "all_models_all_seed_subgroup_metrics.csv")
gaps = pd.read_csv(TABLES / "bosque_light_dark_bootstrap_comparison_readable.csv")

display(overall.head())
display(subgroups.head())
display(gaps.head())

## 3. Operational definition

For a model, metric, and target evidence source, define the **Predictive Representativity deviation** as:

\[
D_{PR}(M, T) = |M_{target} - M_{internal}|
\]

where:

- \(M_{internal}\) is the mean internal HAM10000 test performance across seeds,
- \(M_{target}\) is the mean target performance in BOSQUE overall, BOSQUE-dark, or BOSQUE-light.

Lower values indicate that the internal validation evidence is closer to the external target evidence for that model-metric pair.

This is an operational audit statistic, not a universal theoretical definition of representativity.

In [ ]:
METRICS = ["accuracy", "precision", "recall", "specificity", "f1", "auc_roc", "auc_pr"]

MODEL_LABELS = {
    "resnet50": "ResNet50",
    "densenet121": "DenseNet121",
    "mobilenetv2": "MobileNetV2",
    "efficientnetv2b0": "EfficientNetV2B0",
    "vgg16": "VGG16",
}

METRIC_LABELS = {
    "accuracy": "Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "specificity": "Specificity",
    "f1": "F1",
    "auc_roc": "AUC-ROC",
    "auc_pr": "AUC-PR",
}

## 4. Compute internal-to-external deviations

In [ ]:
overall_mean = (
    overall
    .groupby(["model", "dataset"])[METRICS]
    .mean()
    .reset_index()
)

subgroup_mean = (
    subgroups
    .groupby(["model", "subgroup"])[METRICS]
    .mean()
    .reset_index()
)

rows = []

for model in sorted(overall["model"].unique()):
    internal_row = overall_mean[
        (overall_mean["model"] == model) &
        (overall_mean["dataset"] == "HAM10000_internal_test")
    ]

    bosque_row = overall_mean[
        (overall_mean["model"] == model) &
        (overall_mean["dataset"] == "BOSQUE_public")
    ]

    if internal_row.empty or bosque_row.empty:
        print("Missing internal or BOSQUE row for", model)
        continue

    internal = internal_row.iloc[0]
    bosque = bosque_row.iloc[0]

    for metric in METRICS:
        rows.append({
            "model": model,
            "target": "BOSQUE_overall",
            "metric": metric,
            "internal": internal[metric],
            "target_value": bosque[metric],
            "signed_difference_target_minus_internal": bosque[metric] - internal[metric],
            "pr_deviation_abs": abs(bosque[metric] - internal[metric]),
        })

    for subgroup_name in ["dark", "light"]:
        sg_row = subgroup_mean[
            (subgroup_mean["model"] == model) &
            (subgroup_mean["subgroup"] == subgroup_name)
        ]

        if sg_row.empty:
            print("Missing subgroup row:", model, subgroup_name)
            continue

        sg = sg_row.iloc[0]

        for metric in METRICS:
            rows.append({
                "model": model,
                "target": f"BOSQUE_{subgroup_name}",
                "metric": metric,
                "internal": internal[metric],
                "target_value": sg[metric],
                "signed_difference_target_minus_internal": sg[metric] - internal[metric],
                "pr_deviation_abs": abs(sg[metric] - internal[metric]),
            })

pr = pd.DataFrame(rows)
pr["model_label"] = pr["model"].map(MODEL_LABELS)
pr["metric_label"] = pr["metric"].map(METRIC_LABELS)

out_path = PR_DIR / "predictive_representativity_deviations.csv"
pr.to_csv(out_path, index=False)
print("Wrote:", out_path.relative_to(ROOT))

display(pr.head(20))

## 5. Compact PR deviation summary

In [ ]:
pr_summary = (
    pr
    .groupby(["target", "metric"])["pr_deviation_abs"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
    .sort_values(["metric", "target"])
)

out_path = PR_DIR / "predictive_representativity_deviation_summary.csv"
pr_summary.to_csv(out_path, index=False)
print("Wrote:", out_path.relative_to(ROOT))

display(pr_summary)

## 6. PR deviation heatmap

In [ ]:
def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.savefig(path.with_suffix(".pdf"), bbox_inches="tight")
    print("Wrote:", path.relative_to(ROOT))
    print("Wrote:", path.with_suffix(".pdf").relative_to(ROOT))
    plt.close()

focus_metrics = ["precision", "recall", "f1", "auc_roc", "auc_pr"]
target_order = ["BOSQUE_overall", "BOSQUE_dark", "BOSQUE_light"]

heat = (
    pr[pr["metric"].isin(focus_metrics)]
    .groupby(["target", "metric"])["pr_deviation_abs"]
    .mean()
    .reset_index()
    .pivot(index="target", columns="metric", values="pr_deviation_abs")
    .loc[target_order, focus_metrics]
)

plt.figure(figsize=(9, 4.2))
im = plt.imshow(heat.values, aspect="auto")
plt.colorbar(im, label="Mean absolute deviation from internal HAM10000")
plt.xticks(range(len(focus_metrics)), [METRIC_LABELS[m] for m in focus_metrics], rotation=35, ha="right")
plt.yticks(range(len(target_order)), ["BOSQUE overall", "BOSQUE dark", "BOSQUE light"])
plt.title("Predictive Representativity deviation by external target")

for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        plt.text(j, i, f"{heat.iloc[i, j]:.3f}", ha="center", va="center", fontsize=9)

savefig(FIGURES / "figure_predictive_representativity_deviation_heatmap.png")
display(heat)

## 7. Model-level PR deviations for AUC-PR and F1

In [ ]:
for metric in ["auc_pr", "f1"]:
    plot_df = pr[pr["metric"] == metric].copy()
    plot_df["target_label"] = plot_df["target"].replace({
        "BOSQUE_overall": "Overall",
        "BOSQUE_dark": "Dark",
        "BOSQUE_light": "Light",
    })

    model_order = ["resnet50", "densenet121", "mobilenetv2", "efficientnetv2b0", "vgg16"]
    target_order_labels = ["Overall", "Dark", "Light"]

    x = np.arange(len(model_order))
    width = 0.25

    plt.figure(figsize=(10, 4.8))

    for k, target_label in enumerate(target_order_labels):
        vals = (
            plot_df[plot_df["target_label"] == target_label]
            .set_index("model")
            .loc[model_order, "pr_deviation_abs"]
            .to_numpy()
        )
        plt.bar(x + (k - 1) * width, vals, width, label=target_label)

    plt.xticks(x, [MODEL_LABELS[m] for m in model_order], rotation=20, ha="right")
    plt.ylabel("Absolute deviation from internal HAM10000")
    plt.title(f"Predictive Representativity deviation for {METRIC_LABELS[metric]}")
    plt.legend(frameon=False)

    savefig(FIGURES / f"figure_predictive_representativity_{metric}_by_target.png")

## 8. Relation to light–dark subgroup gaps

In [ ]:
sig = gaps[gaps["significant_fdr_0_05"]].copy()

display(sig[[
    "model", "metric", "dark", "light", "gap_light_minus_dark",
    "bootstrap_ci_low", "bootstrap_ci_high", "p_value_fdr_bh", "sig_fdr"
]].sort_values(["model", "metric"]))

## 9. Summary statements for reporting

The Predictive Representativity analysis showed that internal HAM10000 validation performance was not equally close to all external BOSQUE targets. Deviations were especially visible when BOSQUE was stratified by skin group. This indicates that aggregate internal validation evidence is insufficient to support homogeneous external performance claims across subgroup-specific Objective Reference Points.

The subgroup bootstrap analysis further showed that several light–dark differences remained significant after Benjamini–Hochberg FDR correction, especially for AUC-PR and precision. This supports the interpretation that the external performance claim is not uniformly transportable across skin-group conditions.

## 10. Files produced

This notebook creates:

```text
outputs/predictive_representativity/predictive_representativity_deviations.csv
outputs/predictive_representativity/predictive_representativity_deviation_summary.csv
outputs/figures/figure_predictive_representativity_deviation_heatmap.png
outputs/figures/figure_predictive_representativity_deviation_heatmap.pdf
outputs/figures/figure_predictive_representativity_auc_pr_by_target.png
outputs/figures/figure_predictive_representativity_auc_pr_by_target.pdf
outputs/figures/figure_predictive_representativity_f1_by_target.png
outputs/figures/figure_predictive_representativity_f1_by_target.pdf
```